# Learned weights — does a trained model beat our hand-chosen weights?

`merchant_ranking_hybrid.ipynb` scores merchants with a weighted sum of percentile ranks. The weights in
it were **chosen by us**, informed by which features predicted realised revenue. That is an
expert-weighted linear model, and the spec's Sprint 4 asks for something stricter:

> *"train a model to predict a certain feature that will be used in rankings"*

This notebook does exactly that, on the same features, and then puts the two side by side.

| | Where the weights come from |
| --- | --- |
| **v2 (hand)** | our judgement, after reading per-feature correlations |
| **Learned linear** | least-squares fit on realised revenue — the machine picks every weight |
| **Gradient boosting** | a non-linear model on the same features, for reference |

**The windows.** Identical to §6 of `merchant_ranking_hybrid.ipynb`, so the comparison is fair.

| | Period | Used for |
| --- | --- | --- |
| **Development** | features 1 Mar 2021 – 28 Feb 2022, label 1 Mar – 31 Aug 2022 | fitting the models |
| **Holdout** | features 1 Mar 2021 – 31 Aug 2022, label 1 – 30 Sep 2022 | scoring all pickers **once** |

Nothing is tuned on September. The models see the development window only; September is opened at
the end to report, exactly as the hand weights were treated.

**What a reader should take from this notebook.** Whichever side wins, we learn something worth
saying. If the learned weights win, the ranking should use them. If the hand weights hold up, that
is evidence our reasoning was sound — and it is a far stronger claim once a fitted model has been
given a fair chance to beat it.

September is reused across exploratory evaluations. These tables do not establish an independent model-selection holdout. Frozen weights are not overwritten by these fits.


In [1]:
K = 300
WEIGHTS_HAND = {
    "estimated_bnpl_revenue": .70, "log_value_growth": .08,
    "transactions_per_customer": .04, "repeat_rate": .03, "stability": .03,
    "active_month_ratio": .02, "customer_spread": .05,
    "customer_income": .03, "customer_irsad": .02,
}


# Resolve imports by repository contents, regardless of notebook location.
import sys
from pathlib import Path
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "scripts" / "process_cleaning.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the project repository")
if str(PROJECT_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))


from pathlib import Path

import numpy as np
import pandas as pd
from pyspark.sql import SparkSession, Window, functions as F

ROOT = PROJECT_ROOT
PROCESSED = ROOT / "data" / "tables" / "processed"

# windows - identical to section 6 of merchant_ranking_hybrid.ipynb
DEV_FEAT_START,  DEV_FEAT_END  = "2021-03-01", "2022-02-28"
DEV_LABEL_START, DEV_LABEL_END = "2022-03-01", "2022-08-31"
HOLD_FEAT_START, HOLD_FEAT_END = "2021-03-01", "2022-08-31"
HOLD_LABEL_START, HOLD_LABEL_END = "2022-09-01", "2022-09-30"

spark = (
    SparkSession.builder.config("spark.sql.session.timeZone", "Etc/UTC").appName("MAST30034 Project 2 - Learned weights")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

transactions = spark.read.parquet(str(PROCESSED / "transactions_clean"))
merchants = spark.read.parquet(str(PROCESSED / "merchants_clean"))
consumers = spark.read.parquet(str(PROCESSED / "consumers_sa2"))

print(f"transactions: {transactions.count():,}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/13 23:26:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


transactions: 14,195,505


## 1. Features rebuilt per window

All feature definitions come from the calendar-window function below: calendar growth blocks, zero-filled monthly sample volatility, and customer profiles restricted to each feature window. Production features cannot be reused for earlier predictions.


In [2]:
import pandas as pd
from pyspark.sql import functions as F

def _build_window_features(
    tx,
    months_in_window,
    early_start,
    early_end_exclusive,
    recent_start,
    recent_end_exclusive,
    window_start,
    window_end,
    *, merchants, consumers,
):
    spark = tx.sparkSession
    tx = tx.filter((F.col("order_datetime") >= F.lit(window_start)) &
                   (F.col("order_datetime") < F.date_add(F.to_date(F.lit(window_end)), 1)))
    core = (
        tx.groupBy("merchant_abn")
        .agg(
            F.sum("dollar_value").alias("gmv"),
            F.count("*").alias("transaction_count"),
            F.countDistinct("user_id").alias("unique_customers"),
            F.countDistinct(F.trunc("order_datetime", "month")).alias("active_months"),
        )
        .withColumn(
            "transactions_per_customer",
            F.col("transaction_count") / F.col("unique_customers"),
        )
        .withColumn(
            "active_month_ratio",
            F.col("active_months") / F.lit(months_in_window),
        )
    )

    per_cust = (
        tx.groupBy("merchant_abn", "user_id")
        .agg(
            F.sum("dollar_value").alias("cust_gmv"),
            F.count("*").alias("cust_orders"),
        )
    )

    conc = (
        per_cust.groupBy("merchant_abn")
        .agg(
            (
                F.sum(F.when(F.col("cust_orders") > 1, 1).otherwise(0))
                / F.count("*")
            ).alias("repeat_rate"),
            F.sum(F.pow("cust_gmv", 2)).alias("_sq"),
            F.sum("cust_gmv").alias("_tot"),
        )
        .withColumn("customer_hhi", F.col("_sq") / F.pow(F.col("_tot"), 2))
        .drop("_sq", "_tot")
    )

    monthly = (
        tx.withColumn("month", F.trunc("order_datetime", "month"))
        .groupBy("merchant_abn", "month")
        .agg(F.sum("dollar_value").alias("monthly_gmv"))
    )

    month_starts = [
        (d.date(),)
        for d in pd.date_range(
            start=pd.Timestamp(window_start).replace(day=1),
            end=pd.Timestamp(window_end).replace(day=1),
            freq="MS",
        )
    ]
    complete_months = spark.createDataFrame(month_starts, ["month"])
    merchant_ids = tx.select("merchant_abn").distinct()

    monthly_complete = (
        merchant_ids.crossJoin(complete_months)
        .join(monthly, ["merchant_abn", "month"], "left")
        .fillna({"monthly_gmv": 0.0})
    )

    stab = (
        monthly_complete.groupBy("merchant_abn")
        .agg(
            F.avg("monthly_gmv").alias("mean_monthly_value"),
            F.stddev("monthly_gmv").alias("sd_monthly_value"),
        )
        .withColumn(
            "value_cv",
            F.when(
                F.col("mean_monthly_value") > 0,
                F.col("sd_monthly_value") / F.col("mean_monthly_value"),
            ),
        )
    )

    def three_month_block(start, end_exclusive, prefix):
        return (
            tx.filter(
                (F.col("order_datetime") >= F.lit(start))
                & (F.col("order_datetime") < F.lit(end_exclusive))
            )
            .groupBy("merchant_abn")
            .agg(
                F.sum("dollar_value").alias(f"{prefix}_3m_value"),
                F.countDistinct(F.trunc("order_datetime", "month")).alias(
                    f"{prefix}_active_months"
                ),
            )
        )

    early = three_month_block(early_start, early_end_exclusive, "early")
    recent = three_month_block(recent_start, recent_end_exclusive, "recent")

    growth = (
        early.join(recent, "merchant_abn", "full")
        .fillna(
            {
                "early_3m_value": 0.0,
                "recent_3m_value": 0.0,
                "early_active_months": 0,
                "recent_active_months": 0,
            }
        )
        .withColumn(
            "log_value_growth",
            F.when(
                (F.col("early_active_months") >= 2)
                & (F.col("recent_active_months") >= 2),
                F.log1p("recent_3m_value") - F.log1p("early_3m_value"),
            ),
        )
    )

    cp = per_cust.join(
        consumers.select(
            "user_id",
            "state",
            "sa2_code",
            "sa2_share",
            "abs_median_household_income_weekly",
            "seifa_irsad_decile",
        ),
        on="user_id",
        how="left",
    )

    has_income = F.col("abs_median_household_income_weekly").isNotNull()
    has_irsad = F.col("seifa_irsad_decile").isNotNull()
    wt = F.coalesce(F.col("sa2_share"), F.lit(1.0))

    base = (
        cp.groupBy("merchant_abn")
        .agg(
            (
                F.sum(
                    F.when(
                        has_income,
                        F.col("abs_median_household_income_weekly") * wt,
                    )
                )
                / F.sum(F.when(has_income, wt))
            ).alias("customer_income"),
            (
                F.sum(F.when(has_irsad, F.col("seifa_irsad_decile") * wt))
                / F.sum(F.when(has_irsad, wt))
            ).alias("customer_irsad"),
            F.avg(F.when(has_income, 1.0).otherwise(0.0)).alias(
                "pct_customers_with_abs"
            ),
        )
    )

    return (
        core.join(conc, "merchant_abn", "left")
        .join(stab.select("merchant_abn", "value_cv"), "merchant_abn", "left")
        .join(growth.select("merchant_abn", "log_value_growth"), "merchant_abn", "left")
        .join(base, "merchant_abn", "left")
        .join(
            merchants.select("merchant_abn", "name", "take_rate"),
            "merchant_abn",
            "left",
        )
        .withColumn(
            "estimated_bnpl_revenue",
            F.col("gmv") * F.col("take_rate") / 100.0,
        )
    )



def window_features(tx, merchants, consumers, window_start, window_end,
                    months_in_window=None, early_start=None, early_end_exclusive=None,
                    recent_start=None, recent_end_exclusive=None):
    start, end = pd.Timestamp(window_start), pd.Timestamp(window_end)
    if start.day != 1 or end != end + pd.offsets.MonthEnd(0) or end < start:
        raise ValueError("Feature windows must contain complete calendar months")
    months = (end.year - start.year) * 12 + end.month - start.month + 1
    if months_in_window is not None and months_in_window != months:
        raise ValueError("Month count must match the complete calendar window")
    return _build_window_features(
        tx, months, early_start or str(start.date()),
        early_end_exclusive or str((start + pd.DateOffset(months=3)).date()),
        recent_start or str((end.replace(day=1) - pd.DateOffset(months=2)).date()),
        recent_end_exclusive or str((end.replace(day=1) + pd.DateOffset(months=1)).date()),
        str(start.date()), str(end.date()), merchants=merchants, consumers=consumers,
    )


def build_features(tx, months_in_window, window_start, window_end):
    return window_features(tx, merchants, consumers, window_start, window_end,
                           months_in_window)


def realised(tx):
    return (tx.groupBy("merchant_abn")
              .agg(F.sum("dollar_value").alias("label_gmv"),
                   F.count("*").alias("label_transactions"))
              .join(merchants.select("merchant_abn", "take_rate"), "merchant_abn", "left")
              .withColumn("realised_revenue", F.col("label_gmv") * F.col("take_rate") / 100.0)
              .select("merchant_abn", "label_transactions", "realised_revenue"))


def window_frame(feat_start, feat_end, label_start, label_end, months):
    feats = build_features(
        transactions.filter(F.col("order_datetime").between(feat_start, feat_end)), months, feat_start, feat_end
    ).filter(F.col("take_rate").isNotNull())
    lab = realised(transactions.filter(F.col("order_datetime").between(label_start, label_end)))
    d = feats.join(lab, "merchant_abn", "left").toPandas()
    d[["label_transactions", "realised_revenue"]] = \
        d[["label_transactions", "realised_revenue"]].fillna(0.0)
    return d


dev = window_frame(DEV_FEAT_START, DEV_FEAT_END, DEV_LABEL_START, DEV_LABEL_END, months=12)
hold = window_frame(HOLD_FEAT_START, HOLD_FEAT_END, HOLD_LABEL_START, HOLD_LABEL_END, months=18)

print(f"development : {len(dev):,} merchants, "
      f"${dev['realised_revenue'].sum()/1e6:,.1f}M realised over 6 months")
print(f"holdout     : {len(hold):,} merchants, "
      f"${hold['realised_revenue'].sum()/1e6:,.1f}M realised over September")

development : 4,019 merchants, $30.3M realised over 6 months
holdout     : 4,026 merchants, $5.5M realised over September


## 2. The same inputs the hand weights see

Both models are given **exactly** what v2's score is built from: each feature as a percentile rank
(0–100), missing values at 50, and rank-space shrinkage towards 50 on revenue and growth with
`k = 300`. Nothing is added and nothing is taken away.

That matters. If the learned model were fed raw dollars while the hand weights saw ranks, a
difference in result would tell us about the input scaling rather than about the weights.

**The target** is `log(1 + realised revenue)`. Revenue is enormously skewed — a handful of merchants
earn a thousand times the median — and a least-squares fit on raw dollars would spend all of its
effort on those few and ignore everyone else.

In [3]:


FEATURES = list(WEIGHTS_HAND)
SHRINK = {"estimated_bnpl_revenue", "log_value_growth"}
K = 300


def design(df):
    """Percentile ranks exactly as merchant_ranking_hybrid builds them."""
    d = df.copy()
    d["stability"] = -d["value_cv"]
    d["customer_spread"] = 1 - d["customer_hhi"]
    shrinkage = d["transaction_count"] / (d["transaction_count"] + K)
    R = pd.DataFrame(
        {c: d[c].rank(pct=True, na_option="keep") * 100 for c in FEATURES}
    ).fillna(50.0)
    for c in SHRINK:
        R[c] = shrinkage * R[c] + (1 - shrinkage) * 50.0
    return R


X_dev = design(dev)
y_dev = np.log1p(dev["realised_revenue"])
X_hold = design(hold)

print(f"X_dev: {X_dev.shape[0]:,} merchants x {X_dev.shape[1]} features")
X_dev.describe().loc[["mean", "min", "max"]].round(1)

X_dev: 4,019 merchants x 9 features


,estimated_bnpl_revenue,log_value_growth,transactions_per_customer,repeat_rate,stability,active_month_ratio,customer_spread,customer_income,customer_irsad
mean,56.8,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0
min,12.7,20.3,22.0,22.0,0.0,0.4,0.3,0.0,0.0
max,99.8,80.8,100.0,100.0,100.0,60.0,100.0,100.0,100.0


## 3. Model 1 — learned linear weights

A least-squares fit. Its coefficients *are* weights on the same ranks, so they can be read beside
the hand-chosen ones directly.

Two notes on reading them:

- **A coefficient may be negative.** That is information, not a bug: it says "higher rank on this
  feature goes with lower revenue, once the others are accounted for". The hand weights could not
  express that, because we only ever assigned positive importance.
- **For ranking, scale does not matter** — only relative size. The coefficients are normalised by
  the sum of their absolute values so they sit beside percentages we can compare.

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold

lin = LinearRegression().fit(X_dev, y_dev)

coef = pd.Series(lin.coef_, index=FEATURES)
w_learned = coef / coef.abs().sum()

comparison = pd.DataFrame({
    "hand weight": pd.Series(WEIGHTS_HAND),
    "learned coefficient": coef,
    "learned weight (normalised)": w_learned,
}).sort_values("hand weight", ascending=False)

print(f"R^2 on the development window: {lin.score(X_dev, y_dev):.3f}\n")
comparison.round(4)

R^2 on the development window: 0.783



/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_


,hand weight,learned coefficient,learned weight (normalised)
estimated_bnpl_revenue,0.70,0.0856,0.4066
log_value_growth,0.08,0.0012,0.0057
customer_spread,0.05,0.0078,0.0370
transactions_per_customer,0.04,-0.0507,-0.2407
repeat_rate,0.03,0.0429,0.2036
stability,0.03,0.0042,0.0201
customer_income,0.03,-0.0005,-0.0025
active_month_ratio,0.02,0.0174,0.0829
customer_irsad,0.02,-0.0002,-0.0009


**Are these coefficients stable, or an accident of this one window?**

Five-fold cross-validation inside the development window refits the model five times on different
subsets. If a weight swings wildly between folds, it is noise and should not be trusted however
good it looks on the full fit.

In [5]:
kf = KFold(n_splits=5, shuffle=True, random_state=0)
folds = []
for tr, _ in kf.split(X_dev):
    m = LinearRegression().fit(X_dev.iloc[tr], y_dev.iloc[tr])
    folds.append(pd.Series(m.coef_, index=FEATURES))

fold_df = pd.DataFrame(folds)
stability = pd.DataFrame({
    "mean": fold_df.mean(),
    "std dev": fold_df.std(),
    "sign flips": (np.sign(fold_df) != np.sign(fold_df.mean())).sum(),
}).round(4)

print("A feature whose sign flips between folds is not a reliable signal.\n")
stability.sort_values("mean", key=abs, ascending=False)

A feature whose sign flips between folds is not a reliable signal.



,mean,std dev,sign flips
estimated_bnpl_revenue,0.0856,0.0006,0
transactions_per_customer,-0.0508,0.0060,0
repeat_rate,0.0430,0.0060,0
active_month_ratio,0.0174,0.0008,0
customer_spread,0.0078,0.0007,0
stability,0.0042,0.0000,0
log_value_growth,0.0012,0.0003,0
customer_income,-0.0005,0.0008,2
customer_irsad,-0.0002,0.0007,2


## 4. Model 2 — gradient boosting

The linear fit is constrained to the same shape as our score: one weight per feature, added up. A
boosted tree model is not, so it can use interactions — "high growth only counts when the customer
base is wide", for instance.

It cannot produce weights, so it enters the comparison as a ranker: merchants are ordered by its
predicted revenue. It is here to answer one question — is our linear shape leaving anything on the
table?

### 3b. The same fit, with weights forced to be positive

The unconstrained fit above does something a business cannot act on: it puts a large **positive**
weight on `repeat_rate` and a large **negative** one on `transactions_per_customer`. Those two
columns measure overlapping things — customers coming back — so least squares is free to pair them
off, one pushing up and the other pulling down, and gain a little accuracy from the difference
between them.

That is a textbook symptom of collinearity. It is legitimate arithmetic and indefensible advice:
nobody can tell a credit committee that a merchant whose customers order *more often* is scored
*worse* for it.

So the fit is repeated with one restriction — **no weight may be negative**. The model must now
choose between overlapping features instead of playing them against each other, and any feature it
cannot use is set to exactly zero. Section 5 scores this version alongside the others, which is the
only way to find out what the restriction costs.

In [6]:
lin_nn = LinearRegression(positive=True).fit(X_dev, y_dev)

coef_nn = pd.Series(lin_nn.coef_, index=FEATURES)
w_nn = coef_nn / coef_nn.sum()

comparison = pd.DataFrame({
    "hand weight": pd.Series(WEIGHTS_HAND),
    "learned, unconstrained": w_learned,
    "learned, non-negative": w_nn,
}).sort_values("hand weight", ascending=False)

print(f"R^2 on the development window : {lin_nn.score(X_dev, y_dev):.3f}"
      f"   (unconstrained: {lin.score(X_dev, y_dev):.3f})")
print(f"features the constraint zeroed: {int((coef_nn == 0).sum())} of {len(FEATURES)}")
print("\nAll three sets of weights, normalised so they sum to 1:\n")
comparison.round(4)

R^2 on the development window : 0.781   (unconstrained: 0.783)
features the constraint zeroed: 4 of 9

All three sets of weights, normalised so they sum to 1:



/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/scipy/optimize/_nnls.py:93: RuntimeWarning: divide by zero encountered in matmul
  x, rnorm, info = _nnls(A, b, maxiter)
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/scipy/optimize/_nnls.py:93: RuntimeWarning: overflow encountered in matmul
  x, rnorm, info = _nnls(A, b, maxiter)
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/scipy/optimize/_nnls.py:93: RuntimeWarning: invalid value encountered in matmul
  x, rnorm, info = _nnls(A, b, maxiter)
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/matthewwihartanto/Library/Python/3.10/lib/p

,hand weight,"learned, unconstrained","learned, non-negative"
estimated_bnpl_revenue,0.70,0.4066,0.7700
log_value_growth,0.08,0.0057,0.0112
customer_spread,0.05,0.0370,0.0001
transactions_per_customer,0.04,-0.2407,0.0000
repeat_rate,0.03,0.2036,0.0000
stability,0.03,0.0201,0.0387
customer_income,0.03,-0.0025,0.0000
active_month_ratio,0.02,0.0829,0.1800
customer_irsad,0.02,-0.0009,0.0000


In [7]:
from sklearn.ensemble import HistGradientBoostingRegressor

gbm = HistGradientBoostingRegressor(
    max_depth=3, learning_rate=0.05, max_iter=400,
    l2_regularization=1.0, random_state=0,
).fit(X_dev, y_dev)

print(f"R^2 on the development window: {gbm.score(X_dev, y_dev):.3f}")
print("(a high number here is expected and proves nothing - September is the real test)")

R^2 on the development window: 0.913
(a high number here is expected and proves nothing - September is the real test)


## 5. September comparison

All five pickers are now scored on the holdout window. The features come from March 2021 – August
2022; the money is what each merchant actually earned in September 2022.

- **capture rate** — of the revenue the best possible 100 merchants earned, what share did this
  picker's 100 collect?
- **Spearman** — correlation of ranks across all merchants.
- **hits** — how many of its 100 were in the realised top 100.

Merchants that stopped trading stay in with zero revenue.

In [8]:
from scipy.stats import spearmanr

perfect = hold.nlargest(100, "realised_revenue")
best_possible = perfect["realised_revenue"].sum()
perfect_abns = set(perfect["merchant_abn"])


def judge(label, series):
    sel = hold.assign(_s=series).nlargest(100, "_s")
    return {
        "picker": label,
        "spearman": spearmanr(series, hold["realised_revenue"])[0],
        "capture rate": sel["realised_revenue"].sum() / best_possible,
        "hits in realised top 100": len(set(sel["merchant_abn"]) & perfect_abns),
        "picks that stopped trading": int((sel["label_transactions"] == 0).sum()),
    }


score_hand = sum(X_hold[c] * w for c, w in WEIGHTS_HAND.items())
score_linear = pd.Series(lin.predict(X_hold), index=hold.index)
score_nnls = pd.Series(lin_nn.predict(X_hold), index=hold.index)
score_gbm = pd.Series(gbm.predict(X_hold), index=hold.index)
score_revenue = hold["estimated_bnpl_revenue"].rank(pct=True) * 100

results = pd.DataFrame([
    judge("hand-chosen weights", score_hand),
    judge("learned linear weights", score_linear),
    judge("learned linear, non-negative", score_nnls),
    judge("gradient boosting", score_gbm),
    judge("baseline: past revenue", score_revenue),
    judge("perfect foresight", hold["realised_revenue"]),
]).set_index("picker")

results.round(3)

/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/matthewwihartanto/Library/Python/3.10/lib/python/site-packages/sklearn/linear_model/_base.py:280: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/matthewwihartanto/Library/Pytho

,spearman,capture rate,hits in realised top 100,picks that stopped trading
picker,,,,
hand-chosen weights,0.900,0.867,73,0
learned linear weights,0.919,0.926,82,0
"learned linear, non-negative",0.919,0.928,83,0
gradient boosting,0.944,0.959,89,0
baseline: past revenue,0.940,1.000,99,0
perfect foresight,1.000,1.000,100,0


In [9]:
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, GREEN, GREY = "#4C78A8", "#F58518", "#54A24B", "#8C8C8C"

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8))

caps = results["capture rate"] * 100
palette = {"hand-chosen weights": ORANGE,
           "learned linear weights": BLUE,
           "learned linear, non-negative": GREEN,
           "gradient boosting": "#72B7B2",
           "baseline: past revenue": GREY,
           "perfect foresight": "#B0B0B0"}
colors = [palette.get(k, GREY) for k in caps.index]
bars = axes[0].barh(caps.index[::-1], caps.values[::-1], color=colors[::-1], height=0.62)
axes[0].bar_label(bars, labels=[f"{v:.1f}%" for v in caps.values[::-1]], padding=4, fontsize=9)
axes[0].set_xlim(0, 112)
axes[0].set_xlabel("% of the revenue a perfect top 100 would have earned")
axes[0].set_title("September holdout - capture rate")

idx = np.arange(len(FEATURES))
axes[1].barh(idx - 0.27, pd.Series(WEIGHTS_HAND)[FEATURES].values, height=0.26,
             color=ORANGE, label="hand-chosen")
axes[1].barh(idx, w_learned[FEATURES].values, height=0.26,
             color=BLUE, label="learned, unconstrained")
axes[1].barh(idx + 0.27, w_nn[FEATURES].values, height=0.26,
             color=GREEN, label="learned, non-negative")
axes[1].axvline(0, color="#555", linewidth=1)
axes[1].set_yticks(idx); axes[1].set_yticklabels(FEATURES, fontsize=8)
axes[1].invert_yaxis()
axes[1].set_xlabel("weight (normalised)")
axes[1].set_title("Where the machine disagrees with us")
axes[1].legend(frameon=False, fontsize=8, loc="lower right")

plt.tight_layout(); plt.show()

/var/folders/09/xnwr_j_x37qfyk85k3lsbpf00000gn/T/ipykernel_27107/1754491427.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. Reading the result

Fill the numbers in from §5 — the shape of the claim should be one of these, and all three are
worth saying out loud:

- **Prefer the non-negative fit over the unconstrained one** whenever their capture rates are close. The unconstrained weights buy their edge with a negative coefficient that cannot be explained to a business reader, and a ranking nobody can explain is not usable however well it scores.
- **The learned weights win clearly.** Then the ranking should adopt them, and the write-up says our
  judgement was close but improvable. Swap `WEIGHTS_HYBRID` in `merchant_ranking_hybrid.ipynb` for the
  normalised coefficients and re-run everything downstream.
- **They are within a point or two of each other.** Then keep the hand weights, because they are
  explainable to a credit committee and a fitted model is not — and say explicitly that the fitted
  alternative was built and measured, not avoided.
- **The hand weights win.** Then we have the strongest version of the argument: a model was given
  the same features and a fair chance, and considered judgement still ordered merchants better on
  unseen data.

**Limitations to state either way**

1. **One development window, one holdout month.** September is a single month and a thin target;
   a merchant with an unusual September moves the numbers more than it should.
2. **The models fit `log(1 + revenue)`, not rank directly.** A model trained to optimise rank
   ordering (learning-to-rank) might do better; that is out of scope here.
3. **A fitted weight can be negative**, which our score's design never contemplated. If a negative
   weight is adopted, it needs a business reading, not just a coefficient.
4. **Neither model sees fraud.** The fraud penalty in `fraud_penalty.ipynb` is applied after
   scoring and is unchanged by anything here.
5. **The comparison is about weights, not about features.** Both sides see the same nine columns, so
   nothing here says whether a different feature set would do better.

In [10]:
OUT_WEIGHTS = PROCESSED / "learned_weights.csv"
OUT_RESULTS = PROCESSED / "learned_vs_hand_results.csv"
OUT_TOP = PROCESSED / "top100_learned.csv"

comparison.to_csv(OUT_WEIGHTS)
results.to_csv(OUT_RESULTS)

# the non-negative fit is the one that could actually be adopted, so its shortlist is the one saved
top_learned = (hold.assign(score_learned=score_nnls)
                   .nlargest(100, "score_learned")
                   .loc[:, ["merchant_abn", "name", "take_rate", "estimated_bnpl_revenue",
                            "transaction_count", "realised_revenue", "score_learned"]])
top_learned.to_csv(OUT_TOP, index=False)

print("written:")
for p in (OUT_WEIGHTS, OUT_RESULTS, OUT_TOP):
    print(f"  {p.name}")

hand_top = set(hold.assign(_s=score_hand).nlargest(100, "_s")["merchant_abn"])
print(f"\nnon-negative learned top 100 vs hand-weight top 100 overlap: "
      f"{len(set(top_learned['merchant_abn']) & hand_top)}/100")

print("\nIf the group adopts these weights, replace WEIGHTS_HYBRID in merchant_ranking_hybrid.ipynb with:")
print({c: round(v, 3) for c, v in w_nn.items() if v > 0})

written:
  learned_weights.csv
  learned_vs_hand_results.csv
  top100_learned.csv

non-negative learned top 100 vs hand-weight top 100 overlap: 87/100

If the group adopts these weights, replace WEIGHTS_HYBRID in merchant_ranking_hybrid.ipynb with:
{'estimated_bnpl_revenue': 0.77, 'log_value_growth': 0.011, 'stability': 0.039, 'active_month_ratio': 0.18, 'customer_spread': 0.0}
